# Building a Custom Model for Anomaly Detection

In [2]:
from snowflake.snowpark import Session
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import logging
from datetime import datetime
from sklearn.ensemble import IsolationForest
from snowflake.ml.registry import Registry

d:\snowflake\venv_snowflake\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

connection_parameters = {
  "account": os.getenv('ACCOUNT'),
  "user": os.getenv('USER'),
  "password": os.getenv('PASSWORD'),
  "role": os.getenv('ROLE'), 
  "warehouse": os.getenv('WAREHOUSE')
}

First, we will load our dataset into Snowflake. This we can do by creating a dataframe and saving it on SF as a table.  
Note: The table will by default saved in the PUBLIC schema.

In [4]:
session = Session.builder.configs(connection_parameters).create()  
session.use_database('anomaly_detection')  

2026-01-13 12:10:17 | INFO | snowflake.connector.connection | Snowflake Connector for Python Version: 3.18.0, Python Version: 3.11.9, Platform: Windows-10-10.0.22631-SP0
2026-01-13 12:10:17 | INFO | snowflake.connector.connection | Connecting to GLOBAL Snowflake domain
2026-01-13 12:10:22 | INFO | snowflake.snowpark.session | Snowpark Session information: 
"version" : 1.44.0,
"python.version" : 3.11.9,
"python.connector.version" : 3.18.0,
"python.connector.session.id" : 3718075878871130,
"os.name" : Windows



In [ ]:
table_name = "RPM_VIB_TEMP"

tables = session.sql(
    f"SHOW TABLES LIKE '{table_name}'"
).collect()


if tables:
    spdf = session.table(table_name)
    df = spdf.to_pandas()
else:
    df = pd.read_csv('../synthetic_data_gen/data/synthetic_data.csv.zip', compression='zip')
    df.set_index('timestamp')
  
    df = df.drop(columns=['anomaly'])
    logging.info(df.head())
    spdf = session.create_dataframe(df)
    spdf.write.save_as_table(table_name, mode="overwrite")

In [6]:
# Train-test split
# We will use 80% of data for training and 20% for testing
split = int(len(df) * 0.8)
train_df = df.drop(columns=['timestamp']).iloc[:split].copy()
test_df =  df.drop(columns=['timestamp']).iloc[split:].copy()


In [9]:
# First, a naive isolation forest
X_train = train_df.drop(columns=['timestamp', 'anomaly'], errors='ignore').copy()
X_test  = test_df.drop(columns=['timestamp', 'anomaly'], errors='ignore').copy()

iso = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42
)
iso.fit(X_train)

,n_estimators,200
,max_samples,'auto'
,contamination,0.01
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [10]:
train_df['anomaly'] = pd.Series(iso.predict(X_train)).map({1:0, -1:1})
test_df['anomaly']  = pd.Series(iso.predict(X_test)).map({1:0, -1:1})

In [11]:
anomalies_train = train_df[train_df['anomaly'] == 1]
anomalies_test = test_df[test_df['anomaly'] == 1]

print("Train anomalies:", anomalies_train.shape[0])
print("Test anomalies:", anomalies_test.shape[0])


Train anomalies: 841
Test anomalies: 0


Lol. I was surpised with the above output.  
Let us try feature engineering next:

### Lag Features  
Lag Features are past values of a variable added as new columns so that the model how the value evolves with time.

In [7]:
# Lags: 1 step (10 min), 6 steps (1 hour), 42 steps (~7 hours), 1008 steps (~1 week)
lags = [1, 6, 42, 1008]

for lag in lags:
    df[f'rpm_lag{lag}'] = df['motor_rpm'].shift(lag)
    df[f'vib_lag{lag}'] = df['vibration'].shift(lag)
    df[f'temp_lag{lag}'] = df['temperature'].shift(lag)


## Time-based features
Encoding the weekly seasonality

In [8]:
df['timestamp'] = pd.to_datetime(df['timestamp'])


df['dayofweek'] = df['timestamp'].dt.dayofweek
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute

df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)

df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)

### Rolling mean features
Rolling mean captures trends and volatility by tracking the mean as a sliding window.

In [9]:
windows = [6, 42, 1008]  # 1 hour, 7 hours, 1 week

for w in windows:
    df[f'rpm_mean_{w}'] = df['motor_rpm'].rolling(w).mean()
    df[f'rpm_std_{w}']  = df['motor_rpm'].rolling(w).std()
    df[f'vib_mean_{w}'] = df['vibration'].rolling(w).mean()
    df[f'vib_std_{w}']  = df['vibration'].rolling(w).std()
    df[f'temp_mean_{w}'] = df['temperature'].rolling(w).mean()
    df[f'temp_std_{w}']  = df['temperature'].rolling(w).std()


Let's try again!

In [10]:
df = df.dropna() 
split = int(len(df) * 0.8)
train_df = df.iloc[:split].copy()
test_df  = df.iloc[split:].copy()

X_train = train_df.drop(columns=['timestamp'], errors='ignore').copy()
X_test  = test_df.drop(columns=['timestamp'], errors='ignore').copy()
X_test

,motor_rpm,vibration,temperature,rpm_lag1,vib_lag1,temp_lag1,rpm_lag6,vib_lag6,temp_lag6,rpm_lag42,...,vib_mean_42,vib_std_42,temp_mean_42,temp_std_42,rpm_mean_1008,rpm_std_1008,vib_mean_1008,vib_std_1008,temp_mean_1008,temp_std_1008
84297,0.041629,0.071931,0.091600,0.040696,0.073495,0.096889,0.047144,0.088500,0.111762,0.062204,...,0.083797,0.011048,0.118647,0.035054,0.046190,0.011147,0.077843,0.024135,0.072490,0.105903
84298,0.041809,0.070573,0.085556,0.041629,0.071931,0.091600,0.040613,0.085870,0.112190,0.058880,...,0.082994,0.010743,0.116752,0.034628,0.046200,0.011138,0.077873,0.024107,0.072682,0.105751
84299,0.043660,0.069551,0.080338,0.041809,0.070573,0.085556,0.042809,0.081250,0.110478,0.051051,...,0.082231,0.010526,0.114675,0.034118,0.046211,0.011130,0.077901,0.024081,0.072868,0.105599
84300,0.039988,0.068080,0.074849,0.043660,0.069551,0.080338,0.040187,0.077328,0.105989,0.051409,...,0.081492,0.010400,0.112427,0.033539,0.046214,0.011128,0.077926,0.024058,0.073048,0.105447
84301,0.039905,0.067312,0.069275,0.039988,0.068080,0.074849,0.040821,0.075424,0.101484,0.048690,...,0.080822,0.010385,0.110069,0.032992,0.046217,0.011127,0.077946,0.024041,0.073220,0.105300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105115,0.053610,0.070704,-0.021684,0.047779,0.065638,-0.027377,0.038047,0.052564,-0.035509,0.055474,...,0.060327,0.010908,0.017161,0.056775,0.043869,0.009767,0.072306,0.020888,0.047957,0.088737
105116,0.053353,0.075623,-0.013112,0.053610,0.070704,-0.021684,0.038732,0.053865,-0.036108,0.046231,...,0.059911,0.009916,0.013766,0.054094,0.043867,0.009764,0.072277,0.020863,0.047782,0.088684
105117,0.050184,0.076880,-0.005026,0.053353,0.075623,-0.013112,0.040397,0.055402,-0.036059,0.044863,...,0.059607,0.009149,0.010559,0.050958,0.043866,0.009763,0.072252,0.020842,0.047612,0.088620
105118,0.055278,0.082254,0.004779,0.050184,0.076880,-0.005026,0.044599,0.059081,-0.034376,0.041686,...,0.059559,0.009020,0.007640,0.047495,0.043865,0.009762,0.072233,0.020824,0.047449,0.088548


In [11]:
iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
iso.fit(X_train)

,n_estimators,200
,max_samples,'auto'
,contamination,0.01
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [12]:
train_df.loc[:, 'anomaly'] = np.where(iso.predict(X_train) == 1, 0, 1)
test_df.loc[:, 'anomaly']  = np.where(iso.predict(X_test) == 1, 0, 1)

In [13]:
anomalies_train = train_df[train_df['anomaly']==1]
anomalies_test = test_df[test_df['anomaly']==1]

print(len(anomalies_train))
print(len(anomalies_test))

833
82


The numbers are fairly realistic now!

### Snowflake's model Registry
For a quick reference:
A Model Registry is a centralized store for ML models combined with version control and metadata tracking. Therefore, it handles versioning, metadata (hyperparams, metrics, training data, etc), lifecycle management (staging, production, archived), and integration in pipelines.

Snowflake provides a model registry. Some popular ones are MLFlow, SageMaker, and TFX.

In [ ]:
ml_reg = Registry(session=session)

# Skipping param version_name would cause automatic versioning
mv = ml_reg.log_model(
    iso, 
    model_name="ANOMALYDETECTION_MODEL",
    sample_input_data=X_train,
    conda_dependencies=['scikit-learn']
)

Logging model: validating model and dependencies...:   0%|          | 0/6 [00:00<?, ?it/s]

2026-01-13 12:13:02 | INFO | snowflake.ml.registry._manager.model_manager | Using non-live commit model version
2026-01-13 12:13:05 | INFO | snowflake.ml.registry._manager.model_parameter_reconciler | Local snowflake-ml-python library has version 1.22.0, which is not available in the Snowflake server, embedding local ML library automatically.
d:\snowflake\venv_snowflake\Lib\site-packages\snowflake\ml\registry\_manager\model_parameter_reconciler.py:72: UserWarning: `relax_version` is not set and therefore defaulted to True. Dependency version constraints relaxed from ==x.y.z to >=x.y, <(x+1). To use specific dependency versions for compatibility, reproducibility, etc., set `options={'relax_version': False}` when logging the model.
  reconciled_options = self._reconcile_relax_version(reconciled_options, reconciled_target_platforms)
2026-01-13 12:13:05 | INFO | snowflake.ml.registry._manager.model_manager | Start packaging and uploading your model. It might take some time based on the siz

❌ ERROR: Model logging failed.:  50%|█████     | 3/6 [00:04<00:04,  1.41s/it]          ]  


ValueError: (2110) Either of `signatures` or `sample_input_data` must be provided for this kind of model.

In [21]:
ml_reg.show_models()

,created_on,name,model_type,database_name,schema_name,comment,owner,default_version_name,versions,aliases
0,2026-01-12 00:49:37.957000-08:00,ANOMALYDETECTION_MODEL,USER_MODEL,ANOMALY_DETECTION,PUBLIC,The model does anomaly detection on the synthe...,ACCOUNTADMIN,V1,"[""MEAN_WASP_3"",""V1""]","{""DEFAULT"":""V1"",""FIRST"":""V1"",""LAST"":""MEAN_WASP..."


In [16]:
m = ml_reg.get_model("ANOMALYDETECTION_MODEL")
mv = m.version("mean_wasp_3")

X_pred = mv.run(X_test, function_name="predict")

In [17]:
X_pred

,output_feature_0
0,1
1,1
2,1
3,1
4,1
...,...
20818,1
20819,1
20820,1
20821,1


In [18]:
anomalies_test = X_pred[X_pred['output_feature_0']==1]
print(len(anomalies_test))

20741
